# Ataque ALGEN sobre DPED com ruído calibrado

Versão alternativa de [attack_algen_dped.ipynb](attack_algen_dped.ipynb), fixada à branch `robust-metodol`. Gera os exports pré-downstream para ε∈{1, 2, 4, 8}, executa o smoke test, a grade do ataque e o baseline `bert_pretrained`.

Antes de executar, ajuste `RUN_NAME` na célula de configuração e coloque na pasta `DRIVE_CKPT` os arquivos `.pt` do decodificador ALGEN congelado. O JSON referencia `checkpoint_epoch_99.pt`, mas o arquivo disponível informado é `checkpoint_epoch_100.pt`; a célula de configuração mapeia explicitamente esses nomes. Confirme que esse é o peso que deseja avaliar.

O notebook fixa a branch `robust-metodol` porque ela contém o atacante e os helpers de proveniência necessários ao preflight. A branch é validada na célula de clone antes de executar os experimentos.

O comando de geração usa `--embedding_privacy_sweep` e `--delta 1e-5`. O sweep define ε={1,2,4,8}; não passe `--epsilon`, `--no_dp`, `--privacy_utility_tradeoff` nem flags de DP-SGD/downstream. Cada tabela tem seu orçamento individual; liberar as quatro juntas requer composição.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/otavio-hbf/DPED-Sistemas-Hibridos.git'
REPO = '/content/DPED-Sistemas-Hibridos'
BRANCH = 'robust-metodol'

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO], check=True)

subprocess.run(['git', '-C', REPO, 'status', '--short', '--branch'], check=True)

In [ ]:
%cd /content/DPED-Sistemas-Hibridos
!pip install -q -r requirements.txt evaluate rouge-score pot wandb

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

RUN_NAME = 'yiyic_mmarco_english_maxlength32_train150000_batch_size128_lr0.0001_wd0.0001_epochs100'
REPO_CKPT = os.path.join(REPO, 'algen/SIH-ALGEN-main/outputs/google_flan-t5-small', RUN_NAME)
DRIVE_CKPT = '/content/drive/MyDrive/SIH/algen/checkpoint'
CHECKPOINT_FILENAME_MAP = {'checkpoint_epoch_99.pt': 'checkpoint_epoch_100.pt'}
DRIVE_RUNS = '/content/drive/MyDrive/SIH/privEmbeddings'
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DRIVE_RUN_DIR = os.path.join(DRIVE_RUNS, RUN_ID)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

def persist_path(source, destination):
    if not os.path.exists(source):
        raise FileNotFoundError(f'Etapa não produziu o artefato esperado: {source}')
    target = os.path.join(DRIVE_RUN_DIR, destination)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.isdir(source):
        shutil.copytree(source, target, dirs_exist_ok=True)
    else:
        shutil.copy2(source, target)
    print('Salvo no Drive:', target)
    return target

def persist_embedding_exports(task, epsilons, stage):
    for epsilon in epsilons:
        tag = str(epsilon).replace('.', 'p')
        source = os.path.join('results/embeddings', f'glue_{task}_dp-distill_eps{tag}',
                              'dped_student_pre_downstream')
        persist_path(source, os.path.join('exports', stage, f'epsilon_{tag}'))

# Ajusta best_models para apontar aos pesos do decodificador congelado no Drive.
args_path = os.path.join(REPO_CKPT, 'training_args_and_best_models.json')
with open(args_path, encoding='utf-8') as f:
    decoder_config = json.load(f)
decoder_config['best_models'] = [
    [*entry[:-1], os.path.join(
        DRIVE_CKPT,
        CHECKPOINT_FILENAME_MAP.get(os.path.basename(entry[-1]), os.path.basename(entry[-1])),
    )]
    for entry in decoder_config['best_models']
]
missing_models = [entry[-1] for entry in decoder_config['best_models'] if not os.path.isfile(entry[-1])]
if missing_models:
    raise FileNotFoundError(f'Pesos do decodificador ausentes no Drive: {missing_models}')
os.makedirs(DRIVE_CKPT, exist_ok=True)
with open(os.path.join(DRIVE_CKPT, 'training_args_and_best_models.json'), 'w', encoding='utf-8') as f:
    json.dump(decoder_config, f, indent=4)

ALGEN_CKPT = DRIVE_CKPT
ATTACKER = 'algen/SIH-ALGEN-main/src/attacker_dped_sentence.py'
EPSILONS = ['8.0', '4.0', '2.0', '1.0']
TRAIN_SIZES = [1, 10, 30, 100, 300, 500, 1000]
RIDGE_VALUES = [1.0]

def export_dir_for(epsilon):
    tag = epsilon.replace('.', 'p')
    return f'results/embeddings/glue_sst2_dp-distill_eps{tag}/dped_student_pre_downstream'

EXPORT_EPS8 = export_dir_for('8.0')
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
print('ALGEN checkpoint exists:', os.path.isdir(ALGEN_CKPT))

## Sanity test (executar antes da grade principal)

Este teste exercita o pipeline completo com uma amostra pequena de CoLA, 2 teachers, 1 época e batch grande para reduzir o custo da destilação do vocabulário. Ele gera as quatro saídas em `glue_cola_*`, sem sobrescrever os resultados SST-2. Se falhar, corrija o problema antes de iniciar a grade principal. Não use esses artefatos reduzidos nos resultados do ataque.

In [ ]:
subprocess.run([sys.executable, 'main.py', '--dataset', 'glue', '--task', 'cola', '--method', 'dp-distill', '--embedding_privacy_sweep', '--delta', '1e-5', '--num_epochs', '1', '--batch_size', '512', '--num_teachers', '2', '--max_train_samples', '64', '--max_seq_length', '32', '--subset_strategy', 'random', '--subset_seed', '42', '--seed', '42'], check=True)
persist_embedding_exports('cola', [1.0, 2.0, 4.0, 8.0], 'sanity')

In [ ]:
from pathlib import Path
import json
import numpy as np

SANITY_EPSILONS = [1.0, 2.0, 4.0, 8.0]
for epsilon in SANITY_EPSILONS:
    tag = str(epsilon).replace('.', 'p')
    export_dir = Path('results/embeddings') / f'glue_cola_dp-distill_eps{tag}' / 'dped_student_pre_downstream'
    embeddings = np.load(export_dir / 'token_embeddings.npy')
    token_ids = np.load(export_dir / 'token_ids.npy')
    with open(export_dir / 'token_labels.json', encoding='utf-8') as f:
        token_labels = json.load(f)
    with open(export_dir / 'metadata.txt', encoding='utf-8') as f:
        metadata = dict(line.strip().split('=', 1) for line in f if '=' in line)

    assert embeddings.ndim == 2 and np.isfinite(embeddings).all(), f'Invalid vectors for epsilon={epsilon}'
    assert embeddings.shape[0] == len(token_ids) == len(token_labels), f'Vocabulary alignment failed for epsilon={epsilon}'
    assert float(metadata['epsilon']) == epsilon, f'Wrong epsilon metadata for epsilon={epsilon}'
    print(f'PASS epsilon={epsilon}: shape={embeddings.shape}, export={export_dir}')
    del embeddings

## 1. Gerar as tabelas com orçamento calibrado

O treino usa até 16.384 exemplos; `--delta 1e-5` é validado em relação ao tamanho efetivo do conjunto. A grade já executa ε={1,2,4,8}, exporta o vocabulário completo e termina antes de DP-SGD downstream. A saída marca as etapas de treino/agregação dos teachers, o início e a conclusão de cada ε (`[1/4]` a `[4/4]`) e a barra interna da destilação.

In [ ]:
subprocess.run([sys.executable, 'main.py', '--dataset', 'glue', '--task', 'sst2', '--method', 'dp-distill', '--embedding_privacy_sweep', '--delta', '1e-5', '--num_epochs', '5', '--batch_size', '16', '--max_train_samples', '16384', '--subset_strategy', 'random', '--subset_seed', '42', '--seed', '42'], check=True)
persist_embedding_exports('sst2', [float(eps) for eps in EPSILONS], 'research')

In [ ]:
!ls {EXPORT_EPS8}

### Preflight de proveniência

Confere os metadados, os hashes e a distinção das cinco tabelas antes de executar a grade.

In [ ]:
CONDITION_HASHES = None
sys.path.insert(0, os.path.dirname(ATTACKER))
import importlib
for module_name in ('attacker_dped_sentence', 'embedding_provenance', 'run_provenance'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
attacker_module = importlib.import_module('attacker_dped_sentence')
preflight_conditions = getattr(attacker_module, 'preflight_conditions', None)
if not callable(preflight_conditions):
    revision = subprocess.check_output(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], text=True).strip()
    raise RuntimeError(f'Código desatualizado em {ATTACKER} (commit {revision}); atualize o repositório e reexecute esta célula.')
if len(EPSILONS) != 4:
    raise ValueError('O preflight exige quatro condições de epsilon.')
conditions = preflight_conditions({eps: export_dir_for(eps) for eps in EPSILONS})
os.makedirs('results/attack', exist_ok=True)
with open('results/attack/preflight.json', 'w', encoding='utf-8') as f:
    json.dump(conditions, f, indent=2)
persist_path('results/attack/preflight.json', 'results_attack/preflight.json')
CONDITION_HASHES = conditions
for label, provenance in CONDITION_HASHES.items():
    print(label, provenance['table_array_sha256'])

## 2. Smoke test do atacante

Confirma o carregamento do checkpoint, do export ε=8 e a gravação dos resultados antes da grade completa.

In [ ]:
smoke_cmd = [sys.executable, ATTACKER, '--checkpoint_path', ALGEN_CKPT, '--export_dir', EXPORT_EPS8, '--source', 'dped', '--epsilon', '8.0', '--train_samples', '10', '--test_samples', '10', '--ridge', '1.0', '--run_kind', 'smoke', '--output_dir', 'results/attack_smoke/smoke']
smoke_result = subprocess.run(smoke_cmd, text=True, capture_output=True)
print('=== Smoke stdout ===\n' + (smoke_result.stdout or '(vazio)'))
print('=== Smoke stderr ===\n' + (smoke_result.stderr or '(vazio)'))
if smoke_result.returncode:
    raise RuntimeError(f'Smoke test terminou com código {smoke_result.returncode}; consulte o stderr acima.')
persist_path('results/attack_smoke/smoke', 'results_attack_smoke/smoke')

## 3. Grade do ataque e baseline

Executa ALGEN com o decodificador congelado para cada ε, tamanho de alinhamento e valor de ridge. O baseline `bert_pretrained` usa os mesmos tokens do export ε=8 e não recebe um argumento de epsilon.

In [ ]:
if 'CONDITION_HASHES' not in globals() or CONDITION_HASHES is None:
    raise RuntimeError('Execute o preflight de proveniência antes da grade.')
if len(EPSILONS) != 4 or set(CONDITION_HASHES) != {f'dped:{eps}' for eps in EPSILONS} | {'bert_pretrained'}:
    raise RuntimeError('As condições mudaram; execute o preflight novamente.')
test_selection_hash = None
train_selection_hashes = {}

def run_attacker(args, condition):
    global test_selection_hash
    provenance = CONDITION_HASHES[condition]
    command = [sys.executable, ATTACKER, '--checkpoint_path', ALGEN_CKPT, '--run_kind', 'research',
               '--expected_table_sha256', provenance['table_array_sha256'],
               '--expected_token_ids_sha256', provenance['token_ids_array_sha256']]
    command.extend(str(arg) for arg in args)
    if condition.startswith('dped:'):
        command.extend(['--expected_export_provenance_sha256', provenance['export_provenance_sha256']])
    print(' '.join(command))
    subprocess.run(command, check=True)
    output_dir = args[args.index('--output_dir') + 1]
    relative = os.path.relpath(output_dir, 'results/attack')
    if relative == os.pardir or relative.startswith(os.pardir + os.sep):
        raise ValueError(f'Saída de pesquisa fora de results/attack: {output_dir}')
    persist_path(output_dir, os.path.join('results_attack', relative))
    with open(os.path.join(output_dir, 'summary.json'), encoding='utf-8') as f:
        summary = json.load(f)
    selection = summary['sentence_selection']
    test_hash = selection['test_selection_sha256']
    train_hash = selection['train_selection_sha256']
    train_key = (summary['train_samples'], summary['seed'])
    if test_selection_hash is not None and test_selection_hash != test_hash:
        raise ValueError(f'Seleção de teste divergente em {output_dir}; não agregue a grade.')
    if train_key in train_selection_hashes and train_selection_hashes[train_key] != train_hash:
        raise ValueError(f'Seleção de alinhamento divergente em {output_dir}; não agregue a grade.')
    test_selection_hash = test_hash
    train_selection_hashes[train_key] = train_hash

for epsilon in EPSILONS:
    tag = epsilon.replace('.', 'p')
    for n in TRAIN_SIZES:
        for ridge in RIDGE_VALUES:
            run_attacker([
                '--export_dir', export_dir_for(epsilon), '--source', 'dped', '--epsilon', epsilon,
                '--train_samples', n, '--test_samples', 200, '--ridge', ridge,
                '--output_dir', f'results/attack/dped_eps{tag}/n{n}_r{ridge}',
            ], f'dped:{epsilon}')

for n in TRAIN_SIZES:
    for ridge in RIDGE_VALUES:
        run_attacker([
            '--export_dir', EXPORT_EPS8, '--source', 'bert_pretrained',
            '--train_samples', n, '--test_samples', 200, '--ridge', ridge,
            '--output_dir', f'results/attack/bert_pretrained/n{n}_r{ridge}',
        ], 'bert_pretrained')

## 4. Salvar resultados no Drive

Pesquisa e smoke são copiados para pastas distintas no Drive.

In [ ]:
persist_path('results/attack', 'results_attack')
persist_embedding_exports('sst2', [float(eps) for eps in EPSILONS], 'research')
print('Todos os artefatos concluídos estão no Drive em:', DRIVE_RUN_DIR)